# MVP – Classificação da faixa de aluguel de imóveis no Brasil

## 1. Contexto e objetivo
Este notebook usa o dataset **Brazilian Houses to Rent** (Kaggle), com anúncios de aluguel em cinco cidades brasileiras (São Paulo, Rio de Janeiro, Belo Horizonte, Porto Alegre e Campinas).

**Problema:** dadas as características de um imóvel (cidade, área, quartos, banheiros, vagas, andar, aceita animais e mobiliado), classificar o aluguel em uma de três faixas: **barato**, **médio** ou **caro**.

**Utilidade:** dar a um inquilino ou corretor uma referência rápida de em qual faixa de preço um anúncio se encaixa.

**Fonte:** Kaggle – Brazilian Houses to Rent (conferir e citar a licença do dataset).

## 2. Configuração

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

## 3. Carga dos dados
O CSV está hospedado em um repositório do GitHub e é lido pela URL *raw*, sem necessidade de configuração.

> **Atenção:** troque a URL abaixo pela do seu repositório.

In [ ]:
URL = "https://raw.githubusercontent.com/SEU_USUARIO/SEU_REPO/main/houses_to_rent_v2.csv"

df = pd.read_csv(URL)
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe().T

## 4. Definição do alvo (três faixas)
A faixa é criada a partir do valor do aluguel (`rent amount (R$)`), dividido em **tercis**. Assim as três classes ficam equilibradas e as métricas são mais simples de interpretar.

Os limites das faixas são guardados porque serão úteis para explicar o resultado na aplicação.

In [ ]:
COL_ALUGUEL = "rent amount (R$)"

df["faixa"], limites = pd.qcut(
    df[COL_ALUGUEL], q=3, labels=["barato", "medio", "caro"], retbins=True
)
print("Limites das faixas (R$):", np.round(limites, 2))
print(df["faixa"].value_counts())

## 5. Prevenção de vazamento de dados (*data leakage*)
Algumas colunas derivam do próprio aluguel ou o revelam:

- `rent amount (R$)`: é a variável usada para criar o alvo.
- `total (R$)`: soma do aluguel com as demais taxas.
- `fire insurance (R$)`: proporcional ao valor do aluguel.
- `hoa (R$)` (condomínio) e `property tax (R$)` (IPTU): não derivam do aluguel, mas o usuário da aplicação normalmente não os conhece ao consultar uma faixa de preço. Por isso também são removidos (decisão a justificar/discutir no relatório).

Manter essas colunas daria uma acurácia alta, mas irreal.

In [ ]:
COLS_REMOVER = [
    "rent amount (R$)", "total (R$)", "fire insurance (R$)",
    "hoa (R$)", "property tax (R$)",
]
df = df.drop(columns=COLS_REMOVER)
df.columns

## 6. Limpeza
- A coluna `floor` usa `-` para casas (sem andar); será convertida em 0.
- Há *outliers* extremos em `area` e `floor` (possíveis erros de digitação); serão filtrados com limites conservadores.

In [ ]:
df["floor"] = df["floor"].replace("-", 0).astype(int)

print("Nulos por coluna:")
print(df.isnull().sum())
print("Antes do filtro:", df.shape)

df = df[(df["area"] <= 1000) & (df["floor"] <= 60)].copy()
print("Depois do filtro:", df.shape)

## 7. Análise exploratória rápida

In [ ]:
df["faixa"].value_counts().plot(kind="bar", title="Distribuição das faixas")
plt.show()

df[["area", "rooms", "bathroom", "parking spaces", "floor"]].hist(figsize=(12, 6))
plt.show()

pd.crosstab(df["city"], df["faixa"], normalize="index").round(2)

## 8. Separação treino/teste (holdout)
80% para treino e 20% para teste, com estratificação para manter a proporção das faixas. O conjunto de teste só será usado na avaliação final.

In [ ]:
X = df.drop(columns=["faixa"])
y = df["faixa"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print(y_train.value_counts(normalize=True).round(3))

## Próximos passos
- Pré-processamento em *pipeline* (`ColumnTransformer`): one-hot para `city`, `animal` e `furniture`; normalização/padronização para as numéricas.
- Modelos base com validação cruzada: KNN, CART, Naive Bayes e SVM.
- Otimização de hiperparâmetros (GridSearchCV), avaliação final e exportação do pipeline.